# 05 — Set Up Databricks Genie AI/BI Space

**Purpose:** Create and configure a Databricks Genie space on top of
`dev.safety_stock_gold.ss_recommendations` so buyers can ask natural
language questions and get instant SQL-backed answers.

**What this notebook does:**
1. Verifies all required tables exist in Unity Catalog
2. Prints step-by-step instructions for creating the Genie space in the UI
3. Adds rich table/column descriptions via SQL `COMMENT` statements to help Genie generate better SQL
4. Tests the Genie REST API once you paste in the Space ID
5. Outputs the Space ID for use in your `.env` file

**Prerequisite:** Run notebooks 01–04 first.

## 0. Configuration — Edit before running

In [ ]:
CATALOG = 'dev'
SCHEMA  = 'safety_stock_gold'

# ── Paste your Genie Space ID here after completing Step 3 below ──────────────
GENIE_SPACE_ID = ''   # e.g. '01ef1234abcd5678ef90abcdef12'
# ─────────────────────────────────────────────────────────────────────────────

spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f'USE SCHEMA {SCHEMA}')
print(f'\u2705 Using {CATALOG}.{SCHEMA}')

✅ Using dev.safety_stock_gold


## 1. Verify Tables Exist

In [ ]:
# Tables the Genie space will query
GENIE_TABLES = [
    'ss_recommendations',
    'safety_stock_features',
    'materials',
    'buyers',
]

print(f"{'Table':<35} {'Row count':>10}   {'Status'}")
print('-' * 56)
all_ready = True
for t in GENIE_TABLES:
    try:
        count = spark.table(f'{CATALOG}.{SCHEMA}.{t}').count()
        print(f'{t:<35} {count:>10,}   \u2705 Ready')
    except Exception as e:
        print(f'{t:<35} {"ERROR":>10}   \u274c {e}')
        all_ready = False

if not all_ready:
    raise RuntimeError('Some tables are missing. Run notebooks 01-04 first.')

Table                              Row count   Status
------------------------------------------------------
ss_recommendations                       100   ✅ Ready
safety_stock_features                    100   ✅ Ready
materials                                100   ✅ Ready
buyers                                     5   ✅ Ready


## 2. Enrich Table Comments for Genie

Genie reads Unity Catalog table and column comments to understand the domain.
The cells below set detailed comments optimised for Genie's SQL generation.

In [ ]:
# Extra Genie instructions appended to table-level comments
genie_table_instructions = {
    'ss_recommendations': (
        'PRIMARY TABLE FOR GENIE. Use this table to answer all questions about '
        'safety stock recommendations. '
        'current_ss = what SAP holds today. new_ss = what the ML model recommends. '
        'pct_change > 0 means the model recommends increasing safety stock. '
        'driver_1/2/3 explain WHY the model made the recommendation. '
        'confidence_score near 1.0 = high-confidence recommendation. '
        'Filter by buyer_id to see a specific buyer\'s materials. '
        'Filter by abc_class=\'A\' to focus on high-value materials.'
    ),
    'safety_stock_features': (
        'Use this table when the question is about demand patterns, '
        'lead times, or feature-level analysis. '
        'demand_cv > 0.5 means highly variable demand. '
        'lead_time_std > 5 means unreliable suppliers. '
        'Join to ss_recommendations on material_id + plant for combined analysis.'
    ),
}

for table, instruction in genie_table_instructions.items():
    safe = instruction.replace("'", "''")
    spark.sql(f"COMMENT ON TABLE {CATALOG}.{SCHEMA}.{table} IS '{safe}'")
    print(f'\u2705 Genie-optimised comments applied to {table}')

✅ Genie-optimised comments applied to ss_recommendations
✅ Genie-optimised comments applied to safety_stock_features


## 3. Create Genie Space in Databricks UI

> **This step is done in the Databricks UI, not in code.**

Follow these steps:

1. In the left sidebar click **Catalog** → open `dev` → `safety_stock_gold`
2. In the top navigation click **New** → **Genie space** (or go to **AI/BI → Genie**)
3. Fill in the form:
   - **Name:** `Safety Stock Q&A`
   - **Description:** `Ask natural-language questions about safety stock recommendations for Generac materials`
4. Under **Data**, add all four tables:
   - `dev.safety_stock_gold.ss_recommendations` ← **primary table**
   - `dev.safety_stock_gold.safety_stock_features`
   - `dev.safety_stock_gold.materials`
   - `dev.safety_stock_gold.buyers`
5. Under **Instructions**, paste the text from the cell below
6. Under **Sample questions**, add the examples from the cell below
7. Click **Save**
8. Copy the Space ID from the URL: `…/genie/spaces/<SPACE_ID>`
9. Paste it into `GENIE_SPACE_ID` in cell 0 above and in your `.env` file

In [ ]:
GENIE_INSTRUCTIONS = '''
You are a supply chain analyst AI for Generac Power Systems.
You answer questions about safety stock recommendations produced by an ML model.

Key concepts:
- safety stock (SS): buffer inventory held to absorb demand/lead-time uncertainty
- current_ss: the safety stock level currently in SAP
- new_ss: what the ML model recommends
- pct_change: how much the SS changes (positive = increase)
- ABC class: A = high-value, B = medium, C = low-value materials
- demand_cv: coefficient of variation of weekly demand (>0.5 = highly variable)
- lead_time_mean: average supplier lead time in days
- driver_1/2/3: top SHAP features driving the ML recommendation
- confidence_score: model certainty (0-1, higher is better)

Always prefer the ss_recommendations table for recommendation questions.
Join to safety_stock_features for deeper feature analysis.
Format numbers to 2 decimal places. Use ORDER BY for rankings.
'''.strip()

SAMPLE_QUESTIONS = [
    'Which materials have the largest safety stock increase recommended?',
    'Show me all A-class materials where the model recommends decreasing SS',
    'What is the average confidence score by ABC class?',
    'Which buyer has the most materials with SS changes greater than 20%?',
    'Show materials where demand variability is the top driver',
    'What are the top 10 materials by current safety stock level?',
    'How many materials are recommended to increase vs decrease SS?',
    'Show materials where confidence score is below 0.5',
]

print('\u2550' * 55)
print('PASTE THIS INTO GENIE SPACE \u2192 INSTRUCTIONS')
print('\u2550' * 55)
print(GENIE_INSTRUCTIONS)
print()
print('\u2550' * 55)
print('PASTE THESE INTO GENIE SPACE \u2192 SAMPLE QUESTIONS')
print('\u2550' * 55)
for i, q in enumerate(SAMPLE_QUESTIONS, 1):
    print(f'{i}. {q}')

═══════ PASTE THIS INTO GENIE SPACE → INSTRUCTIONS ═══════

You are a supply chain analyst AI for Generac Power Systems.
You answer questions about safety stock recommendations produced by an ML model.

Key concepts:
- safety stock (SS): buffer inventory held to absorb demand/lead-time uncertainty
- current_ss: the safety stock level currently in SAP
- new_ss: what the ML model recommends
- pct_change: how much the SS changes (positive = increase)
- ABC class: A = high-value, B = medium, C = low-value materials
- demand_cv: coefficient of variation of weekly demand (>0.5 = highly variable)
- lead_time_mean: average supplier lead time in days
- driver_1/2/3: top SHAP features driving the ML recommendation
- confidence_score: model certainty (0–1, higher is better)

Always prefer the ss_recommendations table for recommendation questions.
Join to safety_stock_features for deeper feature analysis.
Format numbers to 2 decimal places. Use ORDER BY for rankings.

═════════════════════════════

## 4. Test the Genie REST API

Run this section **after** you have created the Genie space and pasted
the `GENIE_SPACE_ID` into cell 0.

In [ ]:
import requests, time, json

if not GENIE_SPACE_ID:
    print('\u26a0\ufe0f  GENIE_SPACE_ID is empty. Complete Step 3 first, then re-run.')
else:
    DATABRICKS_HOST  = spark.conf.get('spark.databricks.workspaceUrl')
    DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

    headers = {
        'Authorization': f'Bearer {DATABRICKS_TOKEN}',
        'Content-Type':  'application/json',
    }
    base = f'https://{DATABRICKS_HOST}/api/2.0/genie'

    # ── Start a test conversation ──────────────────────────────────────────────
    test_question = 'How many materials are recommended to increase vs decrease safety stock?'
    print(f'Question: {test_question}')
    print('Sending to Genie...')

    resp = requests.post(
        f'{base}/spaces/{GENIE_SPACE_ID}/start-conversation',
        headers=headers,
        json={'content': test_question},
    )
    resp.raise_for_status()
    data          = resp.json()
    conversation_id = data['conversation_id']
    message_id      = data['message_id']
    print(f'conversation_id : {conversation_id}')
    print(f'message_id      : {message_id}')

    # ── Poll until complete ────────────────────────────────────────────────────
    print('Waiting for Genie response', end='')
    for _ in range(30):
        r = requests.get(
            f'{base}/spaces/{GENIE_SPACE_ID}/conversations/{conversation_id}/messages/{message_id}',
            headers=headers,
        )
        msg = r.json()
        if msg.get('status') in ['COMPLETED', 'FAILED', 'CANCELLED']:
            break
        print('.', end='', flush=True)
        time.sleep(2)

    print(f'\nStatus: {msg.get("status")}')

    # ── Show result ───────────────────────────────────────────────────────────
    for att in msg.get('attachments', []):
        if att.get('query'):
            print(f'\nSQL generated by Genie:\n{att["query"].get("query", "")}')
            print(f'\nGenie description:\n{att["query"].get("description", "")}')
        elif att.get('text'):
            print(f'\nGenie answer:\n{att["text"].get("content", "")}')

    print('\n\u2705 Genie API test complete!')

## 5. Save Space ID to Environment

Copy the output of the next cell and add it to your `.env` file.

In [ ]:
if GENIE_SPACE_ID:
    print('Add this to your .env file:')
    print(f'GENIE_SPACE_ID={GENIE_SPACE_ID}')
    print()
    print('Then launch the Streamlit app:')
    print('  streamlit run app/streamlit_app.py')
else:
    print('\u26a0\ufe0f  Complete Step 3 (create space in UI) and paste the GENIE_SPACE_ID above.')

## Next Step

With `GENIE_SPACE_ID` in your `.env`, launch the Streamlit app:
```bash
streamlit run app/streamlit_app.py
```

- **Page 1 — Genie Q&A**: Natural language questions → Genie API → SQL + results
- **Page 2 — SS Recommendations**: Full recommendation table with filters and approval submission
- **Page 3 — Manager Approval**: Approve / reject buyer submissions